# A2. Advanced Networks

The basic Networks tutorial introduced the core idea: a `Network` advances populations, event-based `NetCon`s, optional `NetStim`s, and synaptic mechanisms together. This tutorial goes deeper into performance-sensitive network workflows:

- construction-time device and dtype placement,
- late `NetStim` attachment,
- banked point-process synapses and slot targets,
- shared spike detectors with `threshold=None`,
- batching and mixed-device placement,
- inspecting built `NetCon`s,
- choosing among event-delay inference backends,
- using bitpacked source-spike history for large SNN-style inference,
- steady-state caching,
- and continuous projections.

## Mental model: where network time is spent

A Dendra network has three main pieces of work per timestep:

1. **Source updates**: optional `NetStim` events and presynaptic population state.
2. **Connection updates**: `NetCon` / `ContinuousCon` delivery into synaptic mechanisms.
3. **Population solves**: the membrane / mechanism update for each population.

For small networks, Python overhead and sparse bookkeeping can dominate. For large GPU networks, the dominant questions are usually:

- how many kernels are launched per timestep,
- how regular the kernels are,
- how much delay state is stored,
- and whether large buffers are allocated directly on the final device rather than copied afterward.

In [ ]:
import gc
import time
from contextlib import nullcontext

import dendra as dn
from dendra.models.mod import exp2syn

import torch

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float32"


def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def cleanup_cuda():
    synchronize()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compact_int(n) -> str:
    if n is None:
        return "—"
    n = int(n)
    a = abs(n)
    if a >= 1_000_000_000:
        return f"{n / 1_000_000_000:.2g}B"
    if a >= 1_000_000:
        return f"{n / 1_000_000:.2g}M"
    if a >= 10_000:
        return f"{n / 1_000:.2g}k"
    return f"{n:,}"


def format_bytes(n: int | None) -> str:
    if n is None:
        return "—"
    n = float(n)
    units = ("B", "KiB", "MiB", "GiB", "TiB")
    for unit in units:
        if abs(n) < 1024.0 or unit == units[-1]:
            if unit == "B":
                return f"{int(n)} B"
            if abs(n) < 10.0:
                return f"{n:.2g} {unit}"
            if abs(n) < 100.0:
                return f"{n:.1f} {unit}"
            return f"{n:.0f} {unit}"
        n /= 1024.0


def tensor_bytes(x) -> int:
    if torch.is_tensor(x):
        return int(x.numel() * x.element_size())
    return 0


def compact_shape(shape) -> str:
    if shape is None:
        return "—"
    if torch.is_tensor(shape):
        shape = tuple(shape.shape)
    if not isinstance(shape, tuple):
        try:
            shape = tuple(shape)
        except TypeError:
            return "—"
    if len(shape) == 0:
        return "scalar"
    return "×".join(compact_int(d) for d in shape)


def shape_and_bytes(x) -> str:
    if not torch.is_tensor(x) or x.numel() == 0:
        return "—"
    return f"{compact_shape(tuple(x.shape))} · {format_bytes(tensor_bytes(x))}"


def short_backend(backend) -> str:
    return {
        "dense": "dense",
        "sparse_calendar": "sparse",
        "bitpacked_history": "bitpack",
    }.get(str(backend), str(backend))


def short_text(text, width: int = 28) -> str:
    if text is None or text == "":
        return "—"
    text = str(text)
    return text if len(text) <= width else text[: max(0, width - 1)] + "…"


def short_conn_name(name, width: int = 30) -> str:
    text = str(name)
    text = text.replace("->", "→").replace("~>", "↝")
    text = text.replace(":spike", "")
    text = text.replace("mech_", "")
    return short_text(text, width)


def format_seconds(x) -> str:
    if x is None:
        return "—"
    return f"{float(x):.3g}s"


def maybe_dataframe(rows, columns=None):
    """Return a compact docs-friendly DataFrame when pandas is available."""
    try:
        import pandas as pd

        df = pd.DataFrame(rows)
        if columns is not None and len(df.columns) > 0:
            df = df[[c for c in columns if c in df.columns]]
        return df
    except Exception:
        for row in rows:
            print(row)
        return rows


## Construction-time placement

For large networks, prefer constructing populations, `NetStim`s, weights, delays, and NetCons directly on the final device and dtype. Moving a fully built network later can require large host-to-device copies and transient duplicate allocations.

Use a Dendra context for default construction placement:

```python
with dn.ctx(DEVICE="cuda", DTYPE="float32"):
    ...
```

A `Network` itself can contain populations on different devices, so it should not be thought of as having a single canonical device. Event and continuous connections are normally placed on the **postsynaptic / synapse** device.

```python
net.devices()   # population name -> device
net.dtypes()    # population name -> dtype
```

For mixed CPU/GPU/MPS networks, Dendra schedules independent components accelerator-first within each timestep phase. This queues accelerator work before CPU population work, allowing overlap where PyTorch and the hardware permit it. Cross-device connections still require moving the presynaptic event/gate signal to the postsynaptic/synapse device, so keep strongly coupled populations colocated when possible.

## Late `NetStim` attachment

A `Network` can be cellular-first:

```python
net = dn.Network({"cells": cells})
```

and later receive artificial sources:

```python
ns = dn.NetStim(N=1024)
net.attach_netstim(ns)
```

The constructor form `dn.Network(..., netstim=ns)` remains supported, but `attach_netstim(...)` is useful when you want the same cellular network to be used with intracellular current injection, external event replay, or no artificial input.

Attaching or replacing a `NetStim`, or adding any new connection, invalidates materialized NetCons. Call `net.initialize(dt)` before the next `run(...)`.

## Banked point-process synapses and slot targets

A physical compartment is not always a unique synaptic target. A single dendritic compartment may host many independent AMPA, NMDA, GABA, or glycine contacts. Use `copies=...` to allocate those contacts as local slots inside one vectorized mechanism:

```python
post.dend.insert(exp2syn, alias="exc", copies=30, tau1=0.2, tau2=2.0)

slots = post.dend.slots(post.mech.exc)
```

`slots.local_index` is the synapse-local coordinate system used by `NetCon` delivery. `slots.flat_index` is the physical population-flat compartment index to which each local slot scatters current. Multiple local slots can share the same `flat_index` while keeping independent mechanism state.

`exp2syn` is a point process: its conductance state and event weights are bare numerical values in µS, and its current method returns nA before Dendra normalizes by compartment area. Thus `weight=0.05` below means 0.05 µS; do not multiply it by `dendra.units.uS`. Other synapses define their own receiving-weight contract.

Connect directly to those slots:

```python
net.connect_one_to_one_slots(
    pre_selection,
    post.dend.slots(post.mech.exc, local_index=torch.arange(30)),
    threshold=0.0,
    weight=0.05,  # µS for exp2syn
    delay=1.0,
)
```

or pass the slot object as the target to `connect(...)` / `connect_one_to_one(...)`.

This is the preferred representation for large point-process banks. It preserves per-contact state while reducing both mechanism-object count and NetCon grouping overhead.

## Shared spike detectors

If many outgoing connections use the same presynaptic threshold, detect spikes once and reuse the event variable:

```python
from dendra.models.mod import spikedetect

pre.soma.insert(spikedetect, alias="spikedetect", threshold=-30.0)

net.connect_one_to_one_slots(
    pre.soma,
    target_slots,
    pre_var="mech.spikedetect.spikes",
    threshold=None,
    weight=weight,
    delay=delay,
)
```

`threshold=None` tells Dendra that `pre_var` is already an event/gate signal. The connection skips per-NetCon voltage thresholding and only handles weighting, delay, and delivery.

## Batching slot-targeted networks

`Network.batch(n)` creates `n` independent copies along a leading batch dimension:

```python
net = build_network()
net = net.batch(32)
net.initialize(dt)
net.run(tstop)
```

For slot-targeted synapses, source indices remain in the source-population coordinate frame and target indices remain in the target-synapse local-slot frame. Users should not manually rebase slot indices after batching.

In [ ]:
def build_netstim_projection(
    *,
    backend: str = "dense",
    N: int = 256,
    fanout: int = 8,
    dt: float = 0.025,
    max_delay_ms: float = 25.0,
    seed: int = 1,
    device: str = DEVICE,
    dtype: str = DTYPE,
):
    # Build a small but scalable NetStim -> Population projection.
    #
    # NetStim sources are useful for backend comparisons because the presynaptic
    # event stream is already binary and source-side, making it eligible for the
    # bitpacked_history backend.
    dev = torch.device(device)
    torch_dtype = torch.float32 if str(dtype) in ("float32", "torch.float32") else torch.float64

    with dn.ctx(DEVICE=device, DTYPE=dtype):
        ns = dn.NetStim(N, interval=10.0, noise=1.0, seed=seed)
        cells = dn.Population(C=N, v_init=-65.0)
        cells.insert(exp2syn, tau1=0.2, tau2=2.0)

        net = dn.Network(
            {"cells": cells},
            netcon_delay_backend=backend,
            track_netcon_events=False,
        )
        net.attach_netstim(ns)

        gen = torch.Generator(device=dev).manual_seed(seed)
        pre_idx = torch.arange(N, device=dev, dtype=torch.long).repeat_interleave(fanout)
        post_idx = torch.randint(N, (pre_idx.numel(),), device=dev, generator=gen)

        max_delay_steps = max(1, int(round(max_delay_ms / dt)))
        delay_steps = torch.randint(
            1,
            max_delay_steps + 1,
            (pre_idx.numel(),),
            device=dev,
            generator=gen,
        )
        delays_ms = delay_steps.to(torch_dtype) * float(dt)

        net.connect_one_to_one(
            net.netstim[pre_idx],
            net.cells[:, post_idx],
            net.cells.mech.exp2syn,
            threshold=None,
            weight=0.05,  # µS for exp2syn
            delay=delays_ms,
            allow_multapses=True
        )

    return net

## Inspecting built NetCons

The following helpers summarize the runtime layout of each event-based connection. They are especially useful when deciding whether a network is delay-buffer dominated.

The default table is intentionally compact for the documentation column width. Column meanings:

- `n/S/D`: connections / target synapse elements / delay-buffer depth in steps;
- `buf`, `evt`, and `hist`: tensor shape plus memory for delivery, event queue, and spike history;
- `mem`: total memory shown in those delay-related tensors;
- `bit`: bitpacked eligibility or mode.

Use `summarize_netcons(net, compact=False)` when exploring interactively and you want the full, wide diagnostic table.


In [ ]:
def _netcon_summary_row(name, nc):
    delivery = getattr(nc, "delivery_buffer", None)
    event_queue = getattr(nc, "event_queue", None)
    spike_hist = getattr(nc, "spike_history_packed", None)
    backend = getattr(nc, "delay_backend", "unknown")
    bit_mode = getattr(nc, "_bitpack_mode", None)
    bit_can_use = getattr(nc, "_bitpack_can_use", None)
    bit_reason = getattr(nc, "_bitpack_ineligible_reason", None)

    if backend == "bitpacked_history":
        if bit_can_use:
            bit = f"{bit_mode or 'yes'} ✓"
        else:
            bit = "no: " + short_text(bit_reason, 20)
    else:
        bit = "—"

    state_bytes = tensor_bytes(delivery) + tensor_bytes(event_queue) + tensor_bytes(spike_hist)
    n_conn = int(nc.numel()) if hasattr(nc, "numel") else None
    syn_numel = int(getattr(nc, "_syn_numel", 0))
    max_delay_steps = int(getattr(nc, "max_delay_steps", 0))

    return {
        # Compact columns for docs output.
        "conn": short_conn_name(name),
        "bkd": short_backend(backend),
        "run": "train" if bool(getattr(nc, "training", False)) else "eval",
        "n/S/D": f"{compact_int(n_conn)}/{compact_int(syn_numel)}/{compact_int(max_delay_steps)}",
        "buf": shape_and_bytes(delivery),
        "evt": shape_and_bytes(event_queue),
        "hist": shape_and_bytes(spike_hist),
        "bit": bit,
        "mem": format_bytes(state_bytes),
        # Full diagnostic fields for interactive exploration.
        "name": name,
        "backend": backend,
        "training": bool(getattr(nc, "training", False)),
        "n_connections": n_conn,
        "syn_numel": syn_numel,
        "max_delay_steps": max_delay_steps,
        "delivery_shape": tuple(delivery.shape) if torch.is_tensor(delivery) else None,
        "delivery_bytes": format_bytes(tensor_bytes(delivery)),
        "event_queue_shape": tuple(event_queue.shape) if torch.is_tensor(event_queue) else None,
        "event_queue_bytes": format_bytes(tensor_bytes(event_queue)),
        "spike_history_shape": tuple(spike_hist.shape) if torch.is_tensor(spike_hist) else None,
        "spike_history_bytes": format_bytes(tensor_bytes(spike_hist)),
        "bitpack_mode": bit_mode,
        "bitpack_can_use": bit_can_use,
        "bitpack_reason": bit_reason,
    }


def summarize_netcons(net, *, compact: bool = True):
    rows = [_netcon_summary_row(name, nc) for name, nc in net.synapses.items()]
    if compact:
        return maybe_dataframe(
            rows,
            columns=("conn", "bkd", "run", "n/S/D", "buf", "evt", "hist", "bit", "mem"),
        )
    return maybe_dataframe(
        rows,
        columns=(
            "name",
            "backend",
            "training",
            "n_connections",
            "syn_numel",
            "max_delay_steps",
            "delivery_shape",
            "delivery_bytes",
            "event_queue_shape",
            "event_queue_bytes",
            "spike_history_shape",
            "spike_history_bytes",
            "bitpack_mode",
            "bitpack_can_use",
            "bitpack_reason",
        ),
    )


net = build_netstim_projection(backend="dense", N=128, fanout=4)
net.initialize(dt=0.025)
summarize_netcons(net)


## Backend overview

Dendra event-based connections expose multiple delay-line backends through `netcon_delay_backend`.

| Backend | Main state | Best use case | Notes |
|---|---:|---|---|
| `"dense"` | `[max_delay_steps, syn_numel]` delivery buffer | Training, differentiable delays, dense GPU inference | Highest memory use, most regular GPU path |
| `"sparse_calendar"` | sparse pending chunks + one dense scratch row | Small / sparse CPU inference | Preserves `net_receive(...)`, but Python sparse bookkeeping is usually slow on GPU |
| `"bitpacked_history"` | packed source-spike history + one dense scratch row | Large SNN-style inference | Requires source-level binary event semantics; typically the most memory-efficient GPU inference backend |

```{important}
Training has a separate `train_delay_backend` contract. The fully general `"dense"` training backend supports arbitrary per-connection semantics. For exact source-level projections, `"source_history"` (and the default `"auto"` policy when eligible) uses compact source history while retaining weight and delay gradients: hard source events use packed history, while surrogate spike gradients use floating source-gate history. `netcon_delay_backend="sparse_calendar"` and `"bitpacked_history"` select inference layouts; reinitialize after switching between training and inference.
```

## Comparing backend memory layouts

This cell builds the same NetStim-driven projection under each backend and prints the allocated connection-state shapes. It is deliberately small so it can run on a laptop; increase `N` and `fanout` to see memory scaling.

In [ ]:
def inspect_backend_layouts(backends=("dense", "sparse_calendar", "bitpacked_history")):
    rows = []
    for backend in backends:
        cleanup_cuda()
        try:
            net = build_netstim_projection(backend=backend, N=256, fanout=8)
            net.initialize(dt=0.025)
            for name, nc in net.synapses.items():
                delivery = getattr(nc, "delivery_buffer", None)
                spike_hist = getattr(nc, "spike_history_packed", None)
                event_queue = getattr(nc, "event_queue", None)
                total = tensor_bytes(delivery) + tensor_bytes(spike_hist) + tensor_bytes(event_queue)
                rows.append(
                    {
                        "bkd": short_backend(backend),
                        "conn": short_conn_name(name),
                        "n/S/D": f"{compact_int(int(nc.numel()))}/{compact_int(int(getattr(nc, '_syn_numel', 0)))}/{compact_int(int(nc.max_delay_steps))}",
                        "buf": shape_and_bytes(delivery),
                        "hist": shape_and_bytes(spike_hist),
                        "mem": format_bytes(total),
                    }
                )
        except Exception as exc:
            rows.append({"bkd": short_backend(backend), "error": short_text(repr(exc), 72)})
        finally:
            try:
                del net
            except UnboundLocalError:
                pass
            cleanup_cuda()

    return maybe_dataframe(rows, columns=("bkd", "conn", "n/S/D", "buf", "hist", "mem", "error"))


inspect_backend_layouts()


## Runtime benchmarking scaffold

Backend benchmarks are workload-dependent. The dense backend often wins on GPU when memory is available because its operations are regular and launch-efficient. The bitpacked backend becomes attractive when `max_delay_steps * syn_numel` is too large or when many outgoing connections share a much smaller source event population.

Use the scaffold below to compare backends under the same connectivity.

In [ ]:
def benchmark_backend(
    backend: str,
    *,
    N: int = 1024,
    fanout: int = 16,
    tstop: float = 10.0,
    dt: float = 0.025,
    use_jit: bool = False,
):
    cleanup_cuda()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    net = build_netstim_projection(backend=backend, N=N, fanout=fanout, dt=dt)
    synchronize()
    build_s = time.perf_counter() - t0

    ctx = dn.ctx(JIT=1) if use_jit else nullcontext()

    t0 = time.perf_counter()
    with ctx:
        net.initialize(dt=dt)
        synchronize()
    init_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    with ctx:
        net.run(tstop=tstop)
        synchronize()
    run_s = time.perf_counter() - t0

    peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

    return {
        "bkd": short_backend(backend),
        "N": compact_int(N),
        "F": compact_int(fanout),
        "steps": compact_int(int(tstop / dt)),
        "build": format_seconds(build_s),
        "init": format_seconds(init_s),
        "run": format_seconds(run_s),
        "peak": format_bytes(peak) if peak is not None else "—",
        "net": net,
    }


results = []
for backend in ("dense", "sparse_calendar", "bitpacked_history"):
    try:
        out = benchmark_backend(backend, N=512, fanout=8, tstop=2.0)
        net = out.pop("net")
        results.append(out)
        del net
    except Exception as exc:
        results.append({"bkd": short_backend(backend), "error": short_text(repr(exc), 72)})
    cleanup_cuda()

maybe_dataframe(results, columns=("bkd", "N", "F", "steps", "build", "init", "run", "peak", "error"))


## Dense backend details

The dense backend keeps a circular delivery buffer:

```text
delivery_buffer: [max_delay_steps, syn_numel]
```

Each timestep:

1. reads the current row,
2. calls `syn.net_receive(row.view(*syn.shape_f), netcon)`,
3. clears that row,
4. detects current events,
5. multiplies by weights,
6. writes into future rows according to integer delays.

Dense inference is memory-heavy but GPU-friendly. It is also the natural runtime layout for differentiable training, because differentiable delay interpolation writes into neighboring delay rows.

In [ ]:
dense_net = build_netstim_projection(backend="dense", N=128, fanout=4)
dense_net.initialize(dt=0.025)

nc = next(iter(dense_net.synapses.values()))
print("advance function:", nc.advance.__name__)
print("delivery_buffer:", tuple(nc.delivery_buffer.shape), format_bytes(tensor_bytes(nc.delivery_buffer)))
print("track_events:", getattr(nc, "track_events", None))
print("event_queue present:", hasattr(nc, "event_queue"))

## Sparse calendar backend details

The sparse calendar backend preserves the existing synapse API but stores only pending nonzero deliveries internally:

```text
_sparse_calendar[ring_slot] -> list of (post_idx_chunk, value_chunk)
delivery_buffer: [1, syn_numel]  # dense scratch row for net_receive(...)
```

This can be a large memory win when only a small number of events are pending. It is not usually the best GPU backend because Python-side sparse calendar mutation creates irregular work and host/device synchronization pressure.

In [ ]:
sparse_net = build_netstim_projection(backend="sparse_calendar", N=128, fanout=4)
sparse_net.initialize(dt=0.025)

nc = next(iter(sparse_net.synapses.values()))
print("advance function:", nc.advance.__name__)
print("delivery_buffer:", tuple(nc.delivery_buffer.shape), format_bytes(tensor_bytes(nc.delivery_buffer)))
print("pending calendar buckets:", len(getattr(nc, "_sparse_calendar", {})))

## Bitpacked source-spike history backend

The bitpacked backend stores past **source spikes**, not future per-synapse deliveries:

```text
spike_history_packed: [max_delay_steps, ceil(n_sources_used / 63)] int64
delivery_buffer:      [1, syn_numel]  # dense scratch row
```

On each step, a kernel reconstructs the dense receive payload from:

- packed source history,
- per-connection delay steps,
- per-connection source word / bit mask,
- post indices,
- and current weights.

This is exact only when the NetCon has one binary event stream per unique presynaptic source. Good cases include:

- `NetStim` sources,
- explicit pre-side binary event variables such as `pre_var="mech.my_mech.spikes"`,
- or thresholded variables with source-consistent thresholds across all outgoing edges.

It is not exact for arbitrary per-connection threshold heterogeneity, because then each outgoing connection can have a different event stream.

In [ ]:
bitpack_net = build_netstim_projection(backend="bitpacked_history", N=128, fanout=4)
bitpack_net.initialize(dt=0.025)

nc = next(iter(bitpack_net.synapses.values()))
print("advance function:", nc.advance.__name__)
print("bitpack_can_use:", getattr(nc, "_bitpack_can_use", None))
print("bitpack_mode:", getattr(nc, "_bitpack_mode", None))
print("bitpack_reason:", getattr(nc, "_bitpack_ineligible_reason", None))
print("delivery_buffer:", tuple(nc.delivery_buffer.shape), format_bytes(tensor_bytes(nc.delivery_buffer)))
print("spike_history_packed:", tuple(nc.spike_history_packed.shape), format_bytes(tensor_bytes(nc.spike_history_packed)))

### Bitpack extension diagnostics

On eligible CUDA workloads, Dendra normally attempts the lazy C++/CUDA bitpack extension shown below and falls back to the PyTorch implementation if it is unavailable or a launch fails. The separate `netcon_bitpack_ops_triton` module is retained as an opt-in experimental/reference implementation; NetCon does not select it automatically. This reports the normal extension's cache/build configuration.

In [ ]:
try:
    from dendra.models.networks import netcon_bitpack_ops
    print("available:", netcon_bitpack_ops.is_available())
    if hasattr(netcon_bitpack_ops, "build_info"):
        for key, value in netcon_bitpack_ops.build_info().items():
            print(f"{key}: {value}")
except Exception as exc:
    print("bitpack extension diagnostics unavailable:", repr(exc))

## Eligibility checks

The bitpacked backend should fail early if a connection cannot be reduced to source-level binary spiking. That failure is useful: it tells you the model needs either the dense backend or a different event variable.

For large SNN-style networks, prefer explicit pre-side spike variables:

```python
pre_var="mech.esser_mech.spikes"
threshold=None
```

That says: "the presynaptic mechanism already produced a binary event stream; NetCon should fan it out."

In [ ]:
# Example diagnostic pattern for a model you already built:
for name, nc in bitpack_net.synapses.items():
    print(name)
    print("  eligible:", getattr(nc, "_bitpack_can_use", None))
    print("  mode:    ", getattr(nc, "_bitpack_mode", None))
    print("  reason:  ", getattr(nc, "_bitpack_ineligible_reason", None))

## Training vs inference

The non-dense backends are inference layouts. In training mode, a NetCon should use the dense differentiable path so gradients can flow through weights, spiking surrogates, scheduled events, and differentiable delay interpolation.

A common pattern is:

1. run a long steady-state or warmup in eval mode,
2. cache the state,
3. switch to train mode,
4. initialize from the cached state,
5. run a shorter differentiable window,
6. backpropagate and update parameters,
7. initialize from the cached state again using the updated weights / delays.

In [ ]:
train_net = build_netstim_projection(backend="bitpacked_history", N=128, fanout=4)
train_net.train()
train_net.set_synaptic_diff_config(
    diff_weights=True,
    diff_delays=True,
    diff_spiking=True,
    taps=2,
)

train_net.initialize(dt=0.025)
nc = next(iter(train_net.synapses.values()))
print("requested backend:", nc.delay_backend)
print("training:", nc.training)
print("advance function:", nc.advance.__name__)
print("delivery_buffer shape in training:", tuple(nc.delivery_buffer.shape))

## Steady-state cache workflows

`steady_state(...)` is for warm-starting later simulations. A useful advanced workflow is:

```python
net.steady_state(tstop=long_warmup, dt=coarse_dt)
net.initialize(dt=fine_dt)
net.run(tstop=short_window)
```

Backend-aware caches should be restored by the connection modules themselves. That allows dense, sparse-calendar, bitpacked-history, and continuous connections to package the runtime state they need and rebuild pending deliveries under the **current** `dt`, weights, and delays.

```{important}
If weights or delays changed after the cache was created, call `initialize(..., reinit_weights=True, reinit_delays=True)` so expanded weight and delay tensors are regenerated before cache restoration.
```

In [ ]:
cache_net = build_netstim_projection(backend="bitpacked_history", N=128, fanout=4)

# Long, coarse warmup. Keep this small in the docs notebook.
cache_net.steady_state(tstop=25.0, dt=0.1, progressbar=False)

# Later: shorter, finer simulation initialized from the cached state.
# Use reinit_weights=True / reinit_delays=True when parameters may have changed.
with dn.ctx(JIT=1):
    cache_net.initialize(
        dt=0.025,
        reinit_weights=True,
        reinit_delays=True,
    )
    cache_net.run(tstop=2.0)

print("state cache keys:", list(cache_net._state_cache.keys()))
print("synapse cache keys:", list(cache_net._syn_cache.keys()))

## Training-loop template with cached state

This is the pattern to use when the long warmup is expensive but each training window is short.

The cached population state is a warm start; if you change weights/delays, it is not the exact fixed point of the updated model. But connection pending state can be rebuilt from cached source history using the current weights and delays, which is usually the desired approximation for repeated short training windows.

In [ ]:
# Pseudocode template; adapt the loss and optimizer to your model.

# net = build_large_model(...)
# optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
#
# net.eval()
# net.steady_state(tstop=1000.0, dt=0.1)
#
# for batch in batches:
#     optimizer.zero_grad(set_to_none=True)
#     net.train()
#     net.set_synaptic_diff_config(diff_weights=True, diff_delays=True, diff_spiking=True)
#
#     # Reinitialize from cached state using updated parameter values.
#     net.initialize(dt=0.025, reinit_weights=True, reinit_delays=True)
#     net.run(tstop=25.0, callbacks=[...])
#
#     loss = ...
#     loss.backward()
#     optimizer.step()


### Newly attached `NetStim` or connection has no effect

Structural changes invalidate materialized NetCons. Reinitialize before running:

```python
net.attach_netstim(ns)
net.connect_one_to_one(...)
net.initialize(dt)
net.run(tstop)
```

### Ambiguous target for a banked synapse

If a point-process mechanism has several local slots on one physical compartment, a physical slice does not identify a unique target. Select slots explicitly:

```python
slots = post.dend.slots(post.mech.exc, local_index=[0, 1, 2])
net.connect_one_to_one_slots(pre[:3], slots, ...)
```

## Continuous projections

`ContinuousCon` is the analog counterpart of `NetCon`: it samples a presynaptic variable every step, applies weights and optional integer delays, scatters into a target `ContinuousSynapse`, and calls:

```python
syn.continuous_receive(payload, continuous_con, input=..., reduce=...)
```

Use `connect_continuous(...)` when the presynaptic signal is an analog variable rather than an event stream.

## Summary

The main network-performance levers are:

- build directly on the intended device/dtype;
- use `attach_netstim(...)` when artificial sources should be optional;
- use banked point-process slots for colocated synapse banks;
- use shared `spikedetect` mechanisms and `threshold=None` when many outgoing connections share a threshold;
- batch independent network replicas with `Network.batch(n)` rather than building Python loops of networks;
- leave event tracking off unless debugging;
- use dense for training and delay derivatives;
- use bitpacked source-spike history for large eligible SNN inference;
- use steady-state caches to avoid repeating long warmup simulations;
- and inspect the actual built NetCon buffers before optimizing.

## Backend selection guide

Use this table as a starting point:

| Situation | Suggested backend |
|---|---|
| Need gradients with respect to delays | `"dense"` in training mode |
| High firing rate, GPU memory available | `"dense"` |
| Very sparse event traffic on CPU | `"sparse_calendar"` |
| Large SNN inference with pre-side spike variable | `"bitpacked_history"` |
| Long delays inflate dense memory | `"bitpacked_history"` if eligible; otherwise consider `"sparse_calendar"` on CPU or a custom GPU calendar |
| Need per-connection event introspection | `track_netcon_events=True`, usually with `"dense"` |

```{warning}
`track_netcon_events=True` can be expensive because event introspection is per connection and per delay slot in the dense backend. Leave it disabled for large inference runs unless you are debugging event delivery.
```

## Troubleshooting checklist

### GPU OOM during construction or reinitialization

- Construct directly on the final device/dtype using `dn.ctx(DEVICE=..., DTYPE=...)`.
- Avoid `.cuda().float()` on already-large networks; use construction-time placement or `.to(device="cuda", dtype=torch.float32)` before large NetCons are built.
- Prefer `bitpacked_history` for large source-spike SNN inference when eligible.

### `bitpacked_history` says the connection is ineligible

Inspect:

```python
nc._bitpack_can_use
nc._bitpack_mode
nc._bitpack_ineligible_reason
```

Common fixes:

- use an explicit binary pre-side event variable, e.g. `pre_var="mech.foo.spikes"`;
- ensure thresholds are uniform per source;
- use `"dense"` for connections that truly need per-connection threshold state.

### Sparse calendar is slow on GPU

That is expected for many workloads. It avoids dense delay memory but has irregular Python-side sparse queue mutation. For all-GPU SNN inference, prefer `bitpacked_history` or a custom GPU-resident event backend.

### Cached state does not reflect new weights/delays

Initialize with:

```python
net.initialize(dt=..., reinit_weights=True, reinit_delays=True)
```

That regenerates expanded weight and delay tensors before connection caches rebuild pending state.

## Optional: large model pattern

For a large model builder such as `Yu2024`, the advanced pattern is to construct directly on the target device and choose the backend before NetCons are built:

```python
from dendra_models.models import Yu2024

with dn.ctx(DEVICE="cuda", DTYPE="float32", JIT=1):
    model = Yu2024(
        netcon_delay_backend="bitpacked_history",
        scale=20,
    )
    model.initialize(dt=0.025)
    model.run(tstop=10.0)
```

For SNN-style recurrent projections that use `threshold=None` and a binary mechanism event variable such as `pre_var="mech.esser_mech.spikes"`, the bitpacked backend is usually the first inference backend to try.

## Summary

The main network-performance levers are:

- build directly on the intended device/dtype;
- leave event tracking off unless debugging;
- use dense for training and delay derivatives;
- use bitpacked source-spike history for large eligible SNN inference;
- use steady-state caches to avoid repeating long warmup simulations;
- and inspect the actual built NetCon buffers before optimizing.